# Servidor Único de Inferencia en Google Colab (M1 + M2 + M3 + Pasajeros)
Ejecuta las celdas en orden antes de iniciar la demostración.


In [ ]:
!nvidia-smi -L || echo 'Sin GPU: cambia a GPU T4 en Entorno de ejecucion'


In [ ]:
import os
if not os.path.exists('/content/Control_Peaje'):
    !git clone https://github.com/Adiel05/Control_Peaje.git /content/Control_Peaje
else:
    !git -C /content/Control_Peaje pull
%cd /content/Control_Peaje
!pip -q install -r inferencia/requirements.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/peaje/evidencias /content/drive/MyDrive/peaje/modelos
!python scripts/diagnostico.py > /content/drive/MyDrive/peaje/evidencias/diagnostico_inferencia.txt
!cat /content/drive/MyDrive/peaje/evidencias/diagnostico_inferencia.txt


In [ ]:
import os
from google.colab import userdata
try:
    os.environ['API_KEY'] = userdata.get('INFERENCE_API_KEY')
except Exception:
    os.environ['API_KEY'] = ''
    print('Aviso: INFERENCE_API_KEY no definido en Secrets; el endpoint aceptara peticiones sin clave.')

os.environ['M2_PESOS'] = '/content/drive/MyDrive/peaje/modelos/m2_seg.pt'
os.environ['M3_PESOS'] = '/content/drive/MyDrive/peaje/modelos/m3_pose.pt'
os.environ['PASAJEROS_PESOS'] = '/content/drive/MyDrive/peaje/modelos/pasajeros_seg.pt'
print('Pesos configurados. Si no existen aun, el servidor usara respaldo COCO para M2/pasajeros.')


In [ ]:
import threading, time, uvicorn
from inferencia.servidor import app

PUERTO = 8000
config = uvicorn.Config(app, host='0.0.0.0', port=PUERTO, log_level='warning')
servidor = uvicorn.Server(config)
hilo = threading.Thread(target=servidor.run, daemon=True)
hilo.start()
time.sleep(5)
!curl -s http://127.0.0.1:8000/health | head -c 400


In [ ]:
import re, subprocess, time
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
log_cf = open('/tmp/cloudflared.log', 'w')
proc_cf = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=log_cf, stderr=subprocess.STDOUT)
url_tunel = None
for _ in range(30):
    time.sleep(1)
    txt = open('/tmp/cloudflared.log').read()
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', txt)
    if m:
        url_tunel = m.group(0)
        break
print('Pega esta URL en la pagina Estado de la web:', url_tunel)


In [ ]:
import numpy as np, cv2, requests
assert url_tunel, 'No se obtuvo URL de Cloudflare'
print('GET /health:', requests.get(url_tunel + '/health', timeout=10).json()['status'])
img_dummy = np.zeros((320, 480, 3), dtype=np.uint8)
ok, buf = cv2.imencode('.jpg', img_dummy)
headers = {'X-API-Key': os.environ['API_KEY']} if os.environ.get('API_KEY') else {}
r = requests.post(url_tunel + '/infer', files={'imagen': ('test.jpg', buf.tobytes(), 'image/jpeg')}, data={'modo': 'completo'}, headers=headers, timeout=30)
print('POST /infer status:', r.status_code, 'latencia_ms:', r.json().get('latencia_ms'))


In [ ]:
# Celda keep-alive: mantenla ejecutando durante la grabacion o demo
import time
while True:
    time.sleep(60)
